# Entrainement de nuit — plaques et chute

Lancez tout, fermez le couvercle, revenez demain.

## Ce que fait ce carnet

Il entraine les deux modeles qui manquent a Ciment's Eye, l'un apres l'autre :

| modele | jeu de donnees | images | duree sur T4 |
|---|---|---|---|
| `plate` | License Plate Recognition v11 | 10 125 | environ 50 min |
| `fall` | falling v1 | 9 912 | environ 80 min |

## La seule chose qui compte vraiment ici

Une session Colab gratuite **se coupe**. Sans precaution, une coupure a la 45e
epoque ne laisse rien du tout.

Ce carnet copie donc `best.pt` et `last.pt` sur votre Google Drive **apres
chaque epoque**. Deux consequences :

- meme si la session meurt en pleine nuit, vous avez un modele exploitable,
  celui de la derniere epoque terminee ;
- si vous relancez le carnet, il **reprend ou il en etait** au lieu de
  recommencer.

Vous ne pouvez donc rien perdre de plus qu'une epoque.

## Avant de lancer

`Execution -> Modifier le type d'execution -> T4 GPU`. La premiere cellule le
verifie et s'arrete si le GPU manque — sans lui, l'entrainement prendrait des
dizaines d'heures.

## 1. Verifier le GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "AUCUN GPU. Execution -> Modifier le type d'execution -> T4 GPU, "
        "puis relancez cette cellule."
    )
print("GPU :", torch.cuda.get_device_name(0))
print("memoire :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "Go")

## 2. Installer et brancher le Drive

Une fenetre vous demandera l'autorisation d'acceder a votre Drive. C'est ce qui
permet aux sauvegardes de survivre a une coupure.

In [ ]:
!pip install -q ultralytics roboflow

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/CimentsEye')
DRIVE.mkdir(parents=True, exist_ok=True)
print('sauvegardes dans :', DRIVE)

## 3. La fonction d'entrainement protege

Rien a modifier ici. Cette cellule definit comment entrainer en sauvegardant a
chaque epoque, et comment reprendre apres une coupure.

In [ ]:
import shutil
import time
from pathlib import Path

from ultralytics import YOLO

LOCAL = Path('/content/runs/detect')


def entrainer(nom, data_yaml, epochs, batch=16, patience=15):
    """Entraine un modele en sauvegardant sur le Drive apres chaque epoque."""
    dossier_drive = DRIVE / nom
    dossier_drive.mkdir(parents=True, exist_ok=True)
    dossier_local = LOCAL / nom
    poids_local = dossier_local / 'weights'

    # Reprise : on remet en place ce que la session precedente avait sauve.
    # args.yaml est indispensable — c'est lui qui porte les parametres de la
    # course interrompue, et sans lui Ultralytics ne sait pas quoi reprendre.
    reprise = (dossier_drive / 'last.pt').exists() and (dossier_drive / 'args.yaml').exists()
    if reprise:
        poids_local.mkdir(parents=True, exist_ok=True)
        for fichier in ('last.pt', 'best.pt'):
            if (dossier_drive / fichier).exists():
                shutil.copy(dossier_drive / fichier, poids_local / fichier)
        shutil.copy(dossier_drive / 'args.yaml', dossier_local / 'args.yaml')
        if (dossier_drive / 'results.csv').exists():
            shutil.copy(dossier_drive / 'results.csv', dossier_local / 'results.csv')
        print(f'[{nom}] reprise depuis la sauvegarde du Drive')
    else:
        print(f'[{nom}] demarrage a neuf')

    def sauvegarder(trainer):
        """Appele apres chaque epoque."""
        try:
            for fichier in ('last.pt', 'best.pt'):
                source = Path(trainer.wdir) / fichier
                if source.exists():
                    shutil.copy(source, dossier_drive / fichier)
            for fichier in ('args.yaml', 'results.csv'):
                source = Path(trainer.save_dir) / fichier
                if source.exists():
                    shutil.copy(source, dossier_drive / fichier)
        except Exception as e:
            # Une sauvegarde ratee ne doit jamais interrompre l'entrainement :
            # le Drive peut avoir un hoquet, la prochaine epoque reessaiera.
            print(f'[{nom}] sauvegarde impossible cette epoque : {e}')

    depart = time.time()
    if reprise:
        modele = YOLO(str(poids_local / 'last.pt'))
        modele.add_callback('on_fit_epoch_end', sauvegarder)
        try:
            modele.train(resume=True)
        except Exception as e:
            # Une reprise peut echouer si la sauvegarde a ete coupee en pleine
            # ecriture. On repart alors du debut plutot que de tout perdre.
            print(f'[{nom}] reprise impossible ({e}) — on recommence a neuf')
            modele = YOLO('yolov8n.pt')
            modele.add_callback('on_fit_epoch_end', sauvegarder)
            modele.train(data=data_yaml, epochs=epochs, imgsz=640, batch=batch,
                         patience=patience, project=str(LOCAL), name=nom,
                         exist_ok=True)
    else:
        modele = YOLO('yolov8n.pt')
        modele.add_callback('on_fit_epoch_end', sauvegarder)
        modele.train(data=data_yaml, epochs=epochs, imgsz=640, batch=batch,
                     patience=patience, project=str(LOCAL), name=nom,
                     exist_ok=True)

    minutes = (time.time() - depart) / 60
    meilleur = poids_local / 'best.pt'
    if meilleur.exists():
        shutil.copy(meilleur, dossier_drive / f'ciments_eye_{nom}_best.pt')
    print(f'[{nom}] termine en {minutes:.0f} min')
    return YOLO(str(meilleur)) if meilleur.exists() else None

## 4. Telecharger les deux jeux de donnees

La cle est celle de votre espace Roboflow. Elle est visible dans le depot
GitHub, qui est public : **revoquez-la quand vous aurez fini**
(Roboflow -> Account -> Roboflow Keys), puis regenerez-en une.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key='kQle1ihpBmsqoXROy27k')

# Plaques — une seule classe, License_Plate.
# Version 11 : 10 125 images. Passez a 4 pour 24 242 images et environ 2 h 30
# de plus, si vous preferez un modele plus robuste.
jeu_plaques = (rf.workspace('roboflow-universe-projects')
                 .project('license-plate-recognition-rxg4e')
                 .version(11).download('yolov8'))

# Chute — trois postures : up, bending, down. Seule « down » alertera.
jeu_chute = (rf.workspace('yolo-h6urw')
               .project('falling-zvpqk')
               .version(1).download('yolov8'))

import yaml
for nom, jeu in (('plaques', jeu_plaques), ('chute', jeu_chute)):
    conf = yaml.safe_load(open(f'{jeu.location}/data.yaml'))
    images = len(list(Path(jeu.location, 'train', 'images').glob('*')))
    print(f'{nom:<10} {images:>6} images d entrainement | classes {conf["names"]}')

## 5. Lancer — c'est ici que vous partez dormir

Les deux entrainements s'enchainent. Comptez deux heures et demie environ.

Si Colab se coupe : rouvrez ce carnet, relancez les cellules 1 a 4, puis
celle-ci. Elle reprendra ou elle s'etait arretee.

In [ ]:
modele_plaques = entrainer('plate', f'{jeu_plaques.location}/data.yaml',
                           epochs=40, batch=32)

modele_chute = entrainer('fall', f'{jeu_chute.location}/data.yaml',
                         epochs=50, batch=16)

print('\nles deux entrainements sont termines')

## 6. Les resultats

Le mAP50 dit si le modele est utilisable.

- **plaques** : viser au-dessus de 0,85. En dessous, le cadrage sera trop lache
  et la lecture du numero en patira.
- **chute** : regarder surtout la classe `down`. C'est la seule qui declenche
  une alerte ; les deux autres ne servent qu'a l'en distinguer.

In [ ]:
for nom, modele in (('plate', modele_plaques), ('fall', modele_chute)):
    if modele is None:
        print(f'{nom} : pas de modele produit')
        continue
    m = modele.val()
    print(f'\n=== {nom} ===')
    print('  mAP50    :', round(float(m.box.map50), 3))
    print('  mAP50-95 :', round(float(m.box.map), 3))
    for i, classe in enumerate(m.names.values()):
        try:
            print(f'  {classe:<12} mAP50 {float(m.box.ap50[i]):.3f}')
        except (IndexError, TypeError):
            pass

## 7. Recuperer les modeles

Tout est deja sur votre Drive, dans `MyDrive/CimentsEye/` :

```
CimentsEye/
  plate/ciments_eye_plate_best.pt
  fall/ciments_eye_fall_best.pt
```

Copiez ces deux fichiers dans le dossier `models/` du projet, puis sur la
machine du site :

```
python scripts/export_openvino.py
```

Ajoutez enfin l'entree du modele de plaques dans `config/config.yaml`, section
`models` — celle de `fall` y est deja :

```yaml
  plate:
    file: models/ciments_eye_plate_best_openvino_model
    conf: 0.35
    enabled: true
```

La cellule ci-dessous les telecharge aussi directement sur votre ordinateur, si
vous preferez.

In [ ]:
from google.colab import files

for nom in ('plate', 'fall'):
    fichier = DRIVE / nom / f'ciments_eye_{nom}_best.pt'
    if fichier.exists():
        print(fichier, '-', round(fichier.stat().st_size / 1e6, 1), 'Mo')
        files.download(str(fichier))
    else:
        print(nom, ': fichier absent — l entrainement n a pas abouti')